---
# Instructions

- Exam 1 - Applied is worth **240 points**. 
- Please strive for clarity and organization.
- **AI Usage**:
   - The use of AI to generate complete answers is strictly **prohibited**. 
   - AI may only be used for debugging purposes.
- **Due Date: Friday, October 9, 2026, by 11:59 PM.**

---
# Exercise 1 (130 points)

- Use only `pandas` and `numpy` (plotting is not required).
- Work through the parts in order; later parts build on the DataFrame you create in earlier parts.
- Write your answer in the code cell below each part, and where asked, state your conclusion in a short markdown cell.
- Code must run top-to-bottom after restarting the kernel. Prefer vectorized pandas operations over loops.
- Each part is worth 10 points (130 points total).

## Data description

The file `EV_Adoption_and_Range_Anxiety_Dataset.csv` contains 10,000 survey respondents (potential car buyers) with demographics, charging infrastructure access, and whether they intend to buy an electric vehicle (EV).

| Column | Description |
|---|---|
| `Buyer_ID` | Respondent identifier (e.g., `EV00001`) |
| `Age`, `Gender` | Demographics |
| `Annual_Income_USD` | Annual income in USD |
| `City_Type` | Urban / Suburban / Rural |
| `Daily_Commute_km` | Daily commute distance (km) |
| `Number_of_Cars_Owned` | Cars in household |
| `Current_Car_Type` | Sedan / SUV / Hatchback / Truck |
| `Charging_Stations_Near_Home`, `Charging_Stations_Near_Work` | Count of nearby charging stations |
| `Home_Charging_Possible` | Yes / No |
| `Environmental_Concern_Level` | Scale 1–5 |
| `Subsidy_Available` | Yes / No |
| `Range_Anxiety_Level` | Low / Medium / High |
| `Will_Buy_EV` | Yes / No |


## Part 1: Load and inspect (10 points)

Import `pandas` and `numpy`. Read the CSV file into a DataFrame named `df`. Display (a) its shape, (b) the first 5 rows, and (c) the data type of each column.

In [1]:
import pandas as pd
import numpy as np

#read the CSV file into a DataFrame
df = pd.read_csv('EV_Adoption_and_range_anxiety_Dataset.csv')

# (a) Display the shape
print("Shape of the DataFrame:", df.shape) 

# (b) Display the first 5 rows
print("\nFirst 5 rows:") 
display(df.head()) 

# (c) Display the data type of each column 
print("\nData types of each column:")
print(df.dtypes) 

Shape of the DataFrame: (10000, 15)

First 5 rows:


,Buyer_ID,Age,Gender,Annual_Income_USD,City_Type,Daily_Commute_km,Number_of_Cars_Owned,Current_Car_Type,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Home_Charging_Possible,Environmental_Concern_Level,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,EV00001,56,Male,44906.0,Rural,18.1,1,Truck,1,0,Yes,4.0,Yes,Low,No
1,EV00002,36,Male,88754.0,Rural,5.0,1,SUV,2,3,Yes,1.0,Yes,Low,No
2,EV00003,42,Male,155958.0,Urban,32.2,2,Sedan,14,3,No,2.0,Yes,Medium,Yes
3,EV00004,31,Male,37059.0,Rural,89.9,1,Sedan,2,1,Yes,4.0,No,Low,No
4,EV00005,48,Female,48552.0,Rural,71.0,1,Sedan,2,0,Yes,3.0,Yes,Low,No



Data types of each column:
Buyer_ID                        object
Age                              int64
Gender                          object
Annual_Income_USD              float64
City_Type                       object
Daily_Commute_km               float64
Number_of_Cars_Owned             int64
Current_Car_Type                object
Charging_Stations_Near_Home      int64
Charging_Stations_Near_Work      int64
Home_Charging_Possible          object
Environmental_Concern_Level    float64
Subsidy_Available               object
Range_Anxiety_Level             object
Will_Buy_EV                     object
dtype: object


## Part 2: Missing values (10 points)

Create a DataFrame called `missing_summary` with one row per column that has at least one missing value, containing the columns `n_missing` and `pct_missing` (percentage rounded to 2 decimals), sorted by `pct_missing` in descending order. Also report how many rows have **at least one** missing value.

In [4]:
# calculate counts and percentages of missing values per column
n_missing = df.isna().sum()
pct_missing = (df.isnull().sum() / len(df) * 100).round(2) 

# Create a summary DataFrame
missing_summary = pd.DataFrame({
    'n_missing': n_missing,
    'pct_missing': pct_missing 
}) 

# Filter to only include columns with at least one missing value
missing_summary = missing_summary[missing_summary['n_missing'] > 0] 

# Sort by pct_missing in descending order 
missing_summary = missing_summary.sort_values('pct_missing', ascending=False)

# Display the result
print("Missing values summary:")
display(missing_summary)

# Count rows with at least one missing value
rows_with_missing = df.isnull().any(axis=1).sum()
print(f"\nNumber of rows with at least one missing value: {rows_with_missing}") 


Missing values summary:


,n_missing,pct_missing
Environmental_Concern_Level,184,1.84
Daily_Commute_km,181,1.81
Annual_Income_USD,178,1.78



Number of rows with at least one missing value: 534


## Part 3: Imputation (10 points)

Handle the missing values as follows, **without dropping any rows**:

1. `Annual_Income_USD`: fill with the median income of the respondent's `City_Type`.
2. `Daily_Commute_km`: fill with the median commute of the respondent's `Current_Car_Type`.
3. `Environmental_Concern_Level`: fill with the overall mode.

Confirm that no missing values remain.

In [5]:
# 1. Fill Annual_Income_USD with median income of the respondent's City_Type
df['Annual_Income_USD'] = df.groupby('City_Type')['Annual_Income_USD'].transform(
    lambda x: x.fillna(x.median())
) 

# 2. Fill Daily_Commute_km with median commute of the respondent's Current_Car_Type 
df['Daily_Commute_km'] = df.groupby('Current_Car_Type')['Daily_Commute_km'].transform(
    lambda x: x.fillna(x.median())
) 

# 3. Fill Environmental_Concern_Level with the overall mode
overall_mode = df['Environmental_Concern_Level'].mode()[0]
df['Environmental_Concern_Level'] = df['Environmental_Concern_Level'].fillna(overall_mode)

# Confirm no missing values remain
print("Total missing values remaining:", df.isnull().sum().sum())
print("\nMissing values per column:") 
print(df.isnull().sum())

Total missing values remaining: 0

Missing values per column:
Buyer_ID                       0
Age                            0
Gender                         0
Annual_Income_USD              0
City_Type                      0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Current_Car_Type               0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Home_Charging_Possible         0
Environmental_Concern_Level    0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64


## Part 4: Type conversion and encoding (10 points)

(a) Create `Buy_Flag` as an integer column (1 if `Will_Buy_EV` is `'Yes'`, else 0).
(b) Convert `Range_Anxiety_Level` into an **ordered** categorical with order Low < Medium < High.
(c) Convert `Gender`, `City_Type`, and `Current_Car_Type` to the `category` dtype.
(d) Convert `Environmental_Concern_Level` to integer.
Show `df.dtypes` and the overall proportion of buyers.

In [8]:
# (a) Create Buy_Flag as an integer (1 if 'Yes', else 0)
df['Buy_Flag'] = (df['Will_Buy_EV'] == 'Yes').astype(int) 

# (b) Range_Anxiety_Level as an ORDERED categorical
df['Range_Anxiety_Level'] = pd.Categorical(
    df['Range_Anxiety_Level'],
     categories=['Low', 'Medium', 'High'], 
     ordered=True
)

# (c) Convert Gender, CIty_Type, Current_Car_Type to category dtype
for col in ['Gender', 'City_Type', 'Current_Car_Type']:
    df[col] = df[col].astype('category')

# (d) Environmental_Concern_Level to integer
df['Environmental_Concern_Level'] = df['Environmental_Concern_Level'].astype(int)

# Show results
print("Data types after conversion:")
print(df.dtypes)

print("n\Overall proportion of buyers (Buy_Flag mean):")
print(round(df['Buy_Flag'].mean(), 4)) 

Data types after conversion:
Buyer_ID                         object
Age                               int64
Gender                         category
Annual_Income_USD               float64
City_Type                      category
Daily_Commute_km                float64
Number_of_Cars_Owned              int64
Current_Car_Type               category
Charging_Stations_Near_Home       int64
Charging_Stations_Near_Work       int64
Home_Charging_Possible           object
Environmental_Concern_Level       int64
Subsidy_Available                object
Range_Anxiety_Level            category
Will_Buy_EV                      object
Buy_Flag                          int64
dtype: object
n\Overall proportion of buyers (Buy_Flag mean):
0.175


## Part 5: Feature engineering with binning (10 points)

Create the following new columns:

- `Age_Group` using `pd.cut` with bins `[17, 29, 44, 59, 100]` and labels `['18-29', '30-44', '45-59', '60+']`.
- `Income_Quartile` using `pd.qcut` on `Annual_Income_USD` with labels `['Q1', 'Q2', 'Q3', 'Q4']`.
- `Total_Stations` = stations near home + stations near work.
- `Long_Commute` = `True` if the commute exceeds the 75th percentile of `Daily_Commute_km`.

Display the count of respondents in each `Age_Group` and `Income_Quartile` (in label order).

In [9]:
# Age_Group using pd.cut
df['Age_Group'] = pd.cut(df['Age'], bins=[17, 29, 44, 59, 100], labels=['18-29', '30-44', '45-59', '60+']) 

# Income_Quartile using pd.qcut
df['Income_Quartile'] = pd.qcut(df['Annual_Income_USD'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

# Total_Stations = home + work
df['Total_Stations'] = (df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']) 

# Long_Commute: True if above the 75th percentile
p75 = df['Daily_Commute_km'].quantile(0.75)
df['Long_Commute'] = df['Daily_Commute_km'] > p75 

# Counts in label order
print("Age_Group counts:")
print(df['Age_Group'].value_counts().sort_index) 

print("\nIncome_Quartile counts:")
print(df['Income_Quartile'].value_counts().sort_index()) 


Age_Group counts:
<bound method Series.sort_index of Age_Group
45-59    3354
30-44    3338
60+      2195
18-29    1113
Name: count, dtype: int64>

Income_Quartile counts:
Income_Quartile
Q1    2500
Q2    2529
Q3    2471
Q4    2500
Name: count, dtype: int64


## Part 6: Filtering and boolean indexing (10 points)

Create `subset` containing respondents who are **Urban or Suburban**, are **between 25 and 40 years old (inclusive)**, have `Subsidy_Available == 'Yes'`, **and** have `Home_Charging_Possible == 'Yes'`. Report the number of rows, the mean income, and the proportion who will buy an EV. Solve once with boolean masks (`.loc`) and verify with `.query()`.

In [10]:
# Method 1: Boolean masks with .loc 
mask = (
    df['City_Type'].isin(['Urban', 'Suburban']) &
    df['Age'].between(25, 40) &
    (df['Subsidy_Available'] == 'Yes') &
    (df['Home_Charging_Possible'] == 'Yes')
)

subset = df.loc[mask]

print("--- Boolean mask (.loc) ---")
print("Number of rows:", len(subset))
print("Mean income:", round(subset['Annual_Income_USD'].mean(), 2))
print("Proportion who will buy an EV:", round(subset['Buy_Flag'].mean(), 4)) 

# Method 2: Verify with query()
subset_q=df.query(
    "City_Type in ['Urban', 'Suburban']" 
    "and 24 <= Age <= 40" 
    "and Subsidy_Available == 'Yes'" 
    "and Home_Charging_Possible == 'Yes'"
)

print("\n--- .query() verification ---")
print("Number of rows:", len(subset_q))
print("Mean income:", round(subset_q['Annual_Income_USD'].mean(), 2))
print("Proportion who will buy an EV:", round(subset_q['Buy_Flag'].mean(), 4)) 


--- Boolean mask (.loc) ---
Number of rows: 984
Mean income: 86128.37
Proportion who will buy an EV: 0.3171

--- .query() verification ---
Number of rows: 984
Mean income: 86128.37
Proportion who will buy an EV: 0.3171


## Part 7: Group-by aggregation (10 points)

Using **named aggregation**, build a table `city_summary` indexed by `City_Type` with columns: `n` (respondents), `buy_rate` (mean of `Buy_Flag`), `median_income`, `avg_commute`, and `avg_stations_home`. Round to 3 decimals and sort by `buy_rate` descending. Which city type has the highest buy rate?

In [14]:
# Named aggregation
city_summary = ( 
    df.groupby('City_Type', observed=True).
    agg(
        n=('Buy_Flag', 'size'),
        buy_rate=('Buy_Flag', 'mean'),
        median_income=('Annual_Income_USD', 'median'),
        avg_commute=('Daily_Commute_km', 'mean'),
        avg_stations_home=('Charging_Stations_Near_Home', 'mean')
)
.round(3)
.sort_values('buy_rate', ascending=False) 
    ) 

print(city_summary)
print ("\nCity type with highest buy rate:", city_summary['buy_rate'].idxmax()) 

              n  buy_rate  median_income  avg_commute  avg_stations_home
City_Type                                                               
Rural      1496     0.190        84235.0       40.606              1.024
Suburban   3563     0.177        84778.0       41.058              3.448
Urban      4941     0.169        84757.5       41.257              8.025

City type with highest buy rate: Rural


## Part 8: Multi-key grouping and ranking (10 points)

Compute the buy rate for every combination of `Current_Car_Type` and `Subsidy_Available` (include the group size). Then add a column `rank_within_car` that ranks the subsidy groups within each car type by buy rate (1 = highest). Return the result as a flat DataFrame (no MultiIndex) sorted by car type, then rank.

In [15]:
# Group by two keys, compute buy rate + size 
grouped = (
    df.groupby(['Current_Car_Type', 'Subsidy_Available'], observed=True)
      .agg(
          n=('Buy_Flag', 'size'),
          buy_rate=('Buy_Flag', 'mean')
      )
      .reset_index()   # flattens the MultiIndex into columns
)

# Rank subsidy groups within each car type by buy_rate (1 = highest) 
grouped['rank_within_car'] = (
    grouped.groupby('Current_Car_Type')['buy_rate']
           .rank(method='dense', ascending=False)
           .astype(int)
)

# Sort by car type, then rank 
grouped = grouped.sort_values(
    ['Current_Car_Type', 'rank_within_car']
).reset_index(drop=True)

print(grouped)

  Current_Car_Type Subsidy_Available     n  buy_rate  rank_within_car
0        Hatchback               Yes   922  0.275488                1
1        Hatchback                No   601  0.019967                2
2              SUV               Yes  2101  0.279867                1
3              SUV                No  1385  0.031047                2
4            Sedan               Yes  2364  0.276650                1
5            Sedan                No  1636  0.021394                2
6            Truck               Yes   621  0.246377                1
7            Truck                No   370  0.029730                2


/var/folders/pv/zcbj60bs18g8nc18wwh6gfhc0000gn/T/ipykernel_39185/3903878937.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped.groupby('Current_Car_Type')['buy_rate']


## Part 9: Cross-tabulation and pivot tables (10 points)

(a) Use `pd.crosstab` to show the proportion of respondents in each `Range_Anxiety_Level` **within** each `City_Type` (each column should sum to 1), rounded to 3 decimals.
(b) Use `pivot_table` to show the mean buy rate with `Age_Group` as rows and `Range_Anxiety_Level` as columns, including row/column margins (`'All'`).

In [19]:
# (a) crosstab: Range_Anxiety_Level proportions within each City_Type
ct = pd.crosstab(
    df['City_Type'], 
    df['Range_Anxiety_Level'], 
    normalize='columns'  # each COLUMN sums to 1
).round(3)

print("(a) Proportion of Range_Anxiety_Level within each City_Type:") 
print(ct)

# (b) pivot_table: mean Buy_Flag by Age_Group (rows) x Range_Anxiety_Level (cols) 
pt = df.pivot_table(
    values='Buy_Flag',
    index='Age_Group',
    columns='Range_Anxiety_Level',
    aggfunc='mean', 
    margins=True,  #adds 'All' row & column
    margins_name='All',
    observed=True  # silendes pandas FutureWarning for category dtypes
).round(3)

print("\n(b) Mean buy rate by Age_Group X Range_Anxiety_Level:") 
print(pt)

(a) Proportion of Range_Anxiety_Level within each City_Type:
Range_Anxiety_Level    Low  Medium   High
City_Type                                
Rural                0.160   0.084  0.210
Suburban             0.366   0.292  0.461
Urban                0.474   0.624  0.329

(b) Mean buy rate by Age_Group X Range_Anxiety_Level:
Range_Anxiety_Level    Low  Medium   High    All
Age_Group                                       
18-29                0.205   0.045  0.000  0.175
30-44                0.188   0.065  0.019  0.167
45-59                0.214   0.077  0.000  0.190
60+                  0.189   0.055  0.000  0.165
All                  0.199   0.064  0.006  0.175


## Part 10: Top-N per group (10 points)

Among respondents who **will buy** an EV, return the 3 highest-income respondents within each `City_Type`. Show `Buyer_ID`, `City_Type`, `Annual_Income_USD`, and `Age`, sorted by `City_Type` and income descending, with a clean 0..n-1 index.

In [21]:
# Filter to buyers, sort, then take top 3 per City_Type
top3 = (
    df[df['Buy_Flag'] == 1].sort_values(['City_Type', 'Annual_Income_USD'], ascending=[True, False])
    .groupby('City_Type', observed=True)
    .head(3)
    [['Buyer_ID', 'City_Type', 'Annual_Income_USD', 'Age']]
    .reset_index(drop=True)
)

print(top3) 

  Buyer_ID City_Type  Annual_Income_USD  Age
0  EV03354     Rural           204597.0   39
1  EV06436     Rural           192788.0   61
2  EV00446     Rural           178668.0   40
3  EV01776  Suburban           214972.0   43
4  EV09820  Suburban           206172.0   26
5  EV05392  Suburban           203871.0   36
6  EV03832     Urban           212377.0   39
7  EV03215     Urban           204784.0   45
8  EV00514     Urban           200083.0   52


## Part 11: Outlier detection and string operations (10 points)

(a) Using the IQR rule (1.5 × IQR beyond Q1/Q3), flag outliers in `Daily_Commute_km` in a boolean column `Commute_Outlier`. Report how many outliers there are and their percentage.
(b) Create `Commute_Capped`, which is `Daily_Commute_km` clipped to the IQR bounds (use `.clip`).
(c) Extract the numeric part of `Buyer_ID` as an integer column `ID_Num` using `str.extract`, and verify that `ID_Num` is unique.

In [ ]:
# (a) IQR rule for Daily_Commute_km 
Q1 = df['Daily_Commute_km'].quantile(0.25)
Q3 = df['Daily_Commute_km'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['Commute_Outlier'] = (
    (df['Daily_Commute_km'] < lower_bound) |
    (df['Daily_Commute_km'] > upper_bound)
)

n_outliers = df['Commute_Outlier'].sum()
pct_outliers = round(n_outliers / len(df) * 100, 2)

print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"Lower bound = {lower_bound}, Upper bound = {upper_bound}")
print(f"Number of commute outliers: {n_outliers}")
print(f"Percentage of outliers: {pct_outliers}%")

# (b) Clip Daily_Commute_km to the IQR bounds 
df['Commute_Capped'] = df['Daily_Commute_km'].clip(
    lower=lower_bound,
    upper=upper_bound
)

# (c) Extract numeric part of Buyer_ID 
df['ID_Num'] = df['Buyer_ID'].str.extract(r'(\d+)').astype(int)

print("\nID_Num unique?", df['ID_Num'].is_unique)
print("Sample Buyer_ID → ID_Num:")
print(df[['Buyer_ID', 'ID_Num']].head()) 

## Part 12: Merging tables (10 points)

Create a small lookup DataFrame `city_info` with columns `City_Type`, `Pop_Density_per_km2`, and `Avg_Electricity_Price_USD_kWh` using these values:

| City_Type | Pop_Density_per_km2 | Avg_Electricity_Price_USD_kWh |
|---|---|---|
| Urban | 3500 | 0.18 |
| Suburban | 1200 | 0.15 |
| Rural | 150 | 0.12 |

Add a fourth row for `'Remote'` (density 20, price 0.11) that does **not** appear in `df`. Merge it into `df` so that **every respondent is kept** and verify the row count is unchanged. Then show with a second merge (`how='outer'`, `indicator=True`) that `'Remote'` appears only in the lookup table.

In [22]:
# Build lookup DataFrame
city_info = pd.DataFrame({
    'City_Type': ['Urban', 'Suburban', 'Rural', 'Remote'],
    'Pop_Density_per_km2': [3500, 1200, 150, 20],
    'Avg_Electricity_Price_USD_kWh': [0.18, 0.15, 0.12, 0.11]
}) 
print("city_info:")
display(city_info) 

# Left merge: every respondent kept, row count unchanged
df_merged = df.merge(city_info, on='City_Type', how='left') 
print("\nRow count before merge:", len(df))
print("Row count after merge: ", len(df_merged)) 
print("Row count unchanged?", len(df) == len(df_merged)) 

# Outer merge with indicator to show 'Remote' only in lookup
check = df.merge(city_info, on='City_Type', how='outer', indicator=True) 
print("\nMerge indicator counts:")
print(check['_merge'].value_counts())

print("\nRows only in city_info (i.e., 'Remote'):")
display(check[check['_merge'] == 'right_only'][['City_Type', '_merge']])



city_info:


,City_Type,Pop_Density_per_km2,Avg_Electricity_Price_USD_kWh
0,Urban,3500,0.18
1,Suburban,1200,0.15
2,Rural,150,0.12
3,Remote,20,0.11



Row count before merge: 10000
Row count after merge:  10000
Row count unchanged? True

Merge indicator counts:
_merge
both          10000
right_only        1
left_only         0
Name: count, dtype: int64

Rows only in city_info (i.e., 'Remote'):


,City_Type,_merge
0,Remote,right_only


## Part 13: Reshaping and final report (10 points)

(a) Build a wide table `wide` with `City_Type` as rows, `Current_Car_Type` as columns, and the **mean `Buy_Flag`** as values (rounded to 3 decimals).
(b) Convert `wide` to a **long** format `long` with columns `City_Type`, `Current_Car_Type`, `buy_rate` using `reset_index` and `melt`, sorted by `buy_rate` descending.
(c) Save the final cleaned `df` (all engineered columns) to `EV_cleaned.csv` without the index, then read it back and confirm the shape matches.

In [23]:
# (a) Wide table: City_Type x Current_Car_Type, mean Buy_Flag
wide = df.pivot_table(
    values='Buy_Flag',
    index='City_Type',
    columns='Current_Car_Type',
    aggfunc='mean',
    observed=True
).round(3)

print("(a) Wide table:")
display(wide)

# (b) Long format via reset_index + melt
long = (
    wide.reset_index()
        .melt(
            id_vars='City_Type',
            var_name='Current_Car_Type',
            value_name='buy_rate'
        )
        .sort_values('buy_rate', ascending=False)
        .reset_index(drop=True)
)

print("\n(b) Long format (top 10):")
display(long.head(10))

# (c) Save cleaned df to CSV and verify 
df.to_csv('EV_cleaned.csv', index=False)

df_reloaded = pd.read_csv('EV_cleaned.csv')

print("\n(c) Shape check:")
print("Original df shape:  ", df.shape)
print("Reloaded df shape:  ", df_reloaded.shape)
print("Shapes match?", df.shape == df_reloaded.shape) 

(a) Wide table:


Current_Car_Type,Hatchback,SUV,Sedan,Truck
City_Type,,,,
Rural,0.231,0.187,0.178,0.182
Suburban,0.185,0.189,0.165,0.168
Urban,0.149,0.173,0.176,0.159



(b) Long format (top 10):


,City_Type,Current_Car_Type,buy_rate
0,Rural,Hatchback,0.231
1,Suburban,SUV,0.189
2,Rural,SUV,0.187
3,Suburban,Hatchback,0.185
4,Rural,Truck,0.182
5,Rural,Sedan,0.178
6,Urban,Sedan,0.176
7,Urban,SUV,0.173
8,Suburban,Truck,0.168
9,Suburban,Sedan,0.165



(c) Shape check:
Original df shape:   (10000, 20)
Reloaded df shape:   (10000, 20)
Shapes match? True


---
# Exercise 2: NumPy Basics (110 points)

This exercise is independent of Exercise 1. Use only `numpy`; **do not use Python loops** unless a part says otherwise. Run the setup cell below first and do not modify it. Each part is worth 10 points.

In [24]:
import numpy as np

rng = np.random.default_rng(500)
# Simulated data: 8 customers x 5 monthly spending amounts (USD)
spend = rng.integers(20, 200, size=(8, 5)).astype(float)
spend[2, 3] = np.nan          # one missing value
spend[5, 0] = np.nan          # another missing value
print(spend)

[[138. 122.  93. 173.  85.]
 [136.  37.  94.  47. 104.]
 [191. 167.  31.  nan 142.]
 [161. 155.  98.  42.  64.]
 [109.  99.  75. 181.  46.]
 [ nan 119. 115. 189.  69.]
 [ 70. 177. 149.  66.  97.]
 [165.  40.  41. 188.  97.]]


## Part 1: Array creation (10 points)

Create the following arrays and print each one:

(a) `a`: integers from 10 to 50 (inclusive) in steps of 5.
(b) `b`: 7 evenly spaced numbers from 0 to 1 (inclusive).
(c) `c`: a 3×3 identity matrix.
(d) `d`: a 2×4 array filled with the value 7.5.
(e) `e`: the integers 0–11 reshaped into a 3×4 matrix.

In [25]:
a = np.arange(10, 51, 5)
b = np.linspace(0, 1, 7)
c = np.eye(3)
d = np.full((2, 4), 7.5)
e = np.arange(12).reshape(3, 4)

print("a =", a)
print("b =", b)
print("c =", c)
print("d =", d)
print("e =", e)

a = [10 15 20 25 30 35 40 45 50]
b = [0.         0.16666667 0.33333333 0.5        0.66666667 0.83333333
 1.        ]
c = [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
d = [[7.5 7.5 7.5 7.5]
 [7.5 7.5 7.5 7.5]]
e = [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


## Part 2: Array attributes and dtypes (10 points)

For the array `spend`, report its shape, number of dimensions, total number of elements, and dtype. Then create `spend_int` by converting a copy of `spend` with the missing values replaced by 0 to integers. Show that `spend` itself was not modified (it still has 2 NaNs).

In [26]:
print("Shape:", spend.shape) 
print("Number of dimensions:", spend.ndim)
print("Total elements:", spend.size)
print("Dtpe:", spend.dtype)

# Make a copy first - this protects the original 
spend_int = spend.copy()
spend_int [np.isnan(spend_int)] = 0
spend_int = spend_int.astype(int) 

print("\nspend_int =\n", spend_int)
print("spend_int dtpe:", spend_int.dtype)

# Verify the orginal is unchanged
print("\nspend still has NaNs?", np.isnan(spend).sum(), "NaNs") 

Shape: (8, 5)
Number of dimensions: 2
Total elements: 40
Dtpe: float64

spend_int =
 [[138 122  93 173  85]
 [136  37  94  47 104]
 [191 167  31   0 142]
 [161 155  98  42  64]
 [109  99  75 181  46]
 [  0 119 115 189  69]
 [ 70 177 149  66  97]
 [165  40  41 188  97]]
spend_int dtpe: int64

spend still has NaNs? 2 NaNs


## Part 3: Indexing and slicing (10 points)

Using `spend`, extract and print:

(a) the element in row 4, column 2;
(b) the entire 3rd row;
(c) the last column;
(d) the sub-matrix made of rows 1–3 and columns 0–2 (inclusive of both ends);
(e) every other row (rows 0, 2, 4, 6) using a step slice;
(f) the last two rows in reverse row order.

In [27]:
print ("(a) spend[4,2] =", spend[4,2]) 
print("\n(b) Row 3:\n", spend[2])          # 0-indexed → row index 2 = 3rd row
print("\n(c) Last column:\n", spend[:, -1])
print("\n(d) Rows 1–3, cols 0–2:\n", spend[1:4, 0:3])
print("\n(e) Every other row (0, 2, 4, 6):\n", spend[::2])
print("\n(f) Last two rows in reverse order:\n", spend[-2:][::-1]) 

(a) spend[4,2] = 75.0

(b) Row 3:
 [191. 167.  31.  nan 142.]

(c) Last column:
 [ 85. 104. 142.  64.  46.  69.  97.  97.]

(d) Rows 1–3, cols 0–2:
 [[136.  37.  94.]
 [191. 167.  31.]
 [161. 155.  98.]]

(e) Every other row (0, 2, 4, 6):
 [[138. 122.  93. 173.  85.]
 [191. 167.  31.  nan 142.]
 [109.  99.  75. 181.  46.]
 [ 70. 177. 149.  66.  97.]]

(f) Last two rows in reverse order:
 [[165.  40.  41. 188.  97.]
 [ 70. 177. 149.  66.  97.]]


## Part 4: Boolean masking and np.where (10 points)

(a) Create `spend_clean` as a copy of `spend` where each NaN is replaced by the **mean of its column** (ignoring NaNs). Do not use a loop.
(b) Using `spend_clean`, count how many entries exceed 150 and print those values as a 1-D array.
(c) Create `tier`, an array the same shape as `spend_clean`, containing `'high'` where spending is at least 120 and `'low'` otherwise (use `np.where`).

In [28]:
# (a) Column-wise NaN replacement without a loop 
spend_clean = spend.copy()
col_means = np.nanmean(spend_clean, axis=0)          # column means, ignoring NaNs
nan_mask = np.isnan(spend_clean)                     # True where NaN
spend_clean[nan_mask] = np.take(col_means, np.where(nan_mask)[1])

print("(a) spend_clean =\n", spend_clean)

# (b) Values exceeding 150 
high_vals = spend_clean[spend_clean > 150]
print("\n(b) Number of values > 150:", high_vals.size)
print("Those values:", high_vals)

# (c) Tier array with np.where
tier = np.where(spend_clean >= 120, 'high', 'low')
print("\n(c) tier =\n", tier) 

(a) spend_clean =
 [[138.         122.          93.         173.          85.        ]
 [136.          37.          94.          47.         104.        ]
 [191.         167.          31.         126.57142857 142.        ]
 [161.         155.          98.          42.          64.        ]
 [109.          99.          75.         181.          46.        ]
 [138.57142857 119.         115.         189.          69.        ]
 [ 70.         177.         149.          66.          97.        ]
 [165.          40.          41.         188.          97.        ]]

(b) Number of values > 150: 10
Those values: [173. 191. 167. 161. 155. 181. 189. 177. 165. 188.]

(c) tier =
 [['high' 'high' 'low' 'high' 'low']
 ['high' 'low' 'low' 'low' 'low']
 ['high' 'high' 'low' 'high' 'high']
 ['high' 'high' 'low' 'low' 'low']
 ['low' 'low' 'low' 'high' 'low']
 ['high' 'low' 'low' 'high' 'low']
 ['low' 'high' 'high' 'low' 'low']
 ['high' 'low' 'low' 'high' 'low']]


## Part 5: Vectorized arithmetic and broadcasting (10 points)

Using `spend_clean`:

(a) Add a 5% tax to every entry.
(b) Standardize each **column** (subtract the column mean, divide by the column standard deviation, `ddof=0`) to create `z`. Verify that the column means of `z` are ~0 and the column standard deviations are 1.
(c) Use broadcasting to multiply the columns by the weights `[0.1, 0.2, 0.3, 0.2, 0.2]` and compute each customer's weighted total (a vector of length 8).

In [29]:
# (a) 5% tax
taxed = spend_clean * 1.05
print("(a) after 5% tax:\n", taxed)

# (b) Standardize each column (z-scores, ddof=O)
z = (spend_clean - spend_clean.mean(axis=0)) / spend_clean.std(axis=0, ddof=0)

print("\n(b) Column means of z (should be ~0):", z.mean(axis=0).round(10))
print("Column stds of z (should be ~1):", z.std(axis=0, ddof=0).round(10))

# (c) Weighted totals via broadcasting
weights = np.array([0.1, 0.2, 0.3, 0.2, 0.2])
weighted_total = (spend_clean * weights).sum(axis=1)

print("\n(c) Weights sum =", weights.sum())   # sanity check: should be 1.0
print("Weighted totals:\n", weighted_total)

(a) after 5% tax:
 [[144.9  128.1   97.65 181.65  89.25]
 [142.8   38.85  98.7   49.35 109.2 ]
 [200.55 175.35  32.55 132.9  149.1 ]
 [169.05 162.75 102.9   44.1   67.2 ]
 [114.45 103.95  78.75 190.05  48.3 ]
 [145.5  124.95 120.75 198.45  72.45]
 [ 73.5  185.85 156.45  69.3  101.85]
 [173.25  42.    43.05 197.4  101.85]]

(b) Column means of z (should be ~0): [ 0. -0. -0.  0.  0.]
Column stds of z (should be ~1): [1. 1. 1. 1. 1.]

(c) Weights sum = 1.0
Weighted totals:
 [117.7         79.4        115.51428571  97.7         98.6
 123.75714286 119.7         93.8       ]


## Part 6: Aggregations along axes (10 points)

Using `spend_clean`, compute and print:

(a) the overall mean, median, and standard deviation;
(b) the total spending per customer (row sums);
(c) the maximum spending in each month (column maxima);
(d) the index of the customer with the highest total spending (`argmax`);
(e) the month (column index) in which each customer spent the most.

In [30]:
# (a) Overall stats 
print("(a) Overall mean:", round(spend_clean.mean(), 2))
print("    Overall median:", round(np.median(spend_clean), 2))
print("    Overall std:", round(spend_clean.std(), 2))

# (b) Total spending per customer (row sums)
row_sums = spend_clean.sum(axis=1)
print("\n(b) Total per customer:\n", row_sums)

# (c) Max spending per month (column maxima)
col_max = spend_clean.max(axis=0)
print("\n(c) Max per month:\n", col_max)

# (d) Customer with highest total spending
top_customer = np.argmax(row_sums)
print("\n(d) Customer index with highest total:", top_customer)
print("    Their total:", row_sums[top_customer])

# (e) Month (column index) of each customer's max spend
best_months = np.argmax(spend_clean, axis=1)
print("\n(e) Best month per customer:\n", best_months)

(a) Overall mean: 110.93
    Overall median: 106.5
    Overall std: 48.19

(b) Total per customer:
 [611.         418.         657.57142857 520.         510.
 630.57142857 559.         531.        ]

(c) Max per month:
 [191. 177. 149. 189. 142.]

(d) Customer index with highest total: 2
    Their total: 657.5714285714286

(e) Best month per customer:
 [3 0 0 0 3 3 1 3]


## Part 7: Random numbers and reproducibility (10 points)

(a) Create a generator with seed 2026 and draw 1,000 values from a normal distribution with mean 50 and standard deviation 10 (`x`). Report the sample mean and standard deviation.
(b) Re-create the generator with the same seed and show that the draws are identical.
(c) From `x`, randomly choose 10 values **without replacement** and print them.
(d) Simulate 20 fair coin flips (0/1) using `integers` and report the number of heads.

In [32]:
# (a) Draw 1000 samples from N(50, 10) with seed 2026
rng = np.random.default_rng(2026)
x = rng.normal(loc=50, scale=10, size=1000)

print("(a) Sample mean:", round(x.mean(), 4))
print("    Sample std: ", round(x.std(), 4))

# (b) Same seed → identical draws
rng2 = np.random.default_rng(2026)
x2 = rng2.normal(loc=50, scale=10, size=1000)

print("\n(b) Identical to first draw?", np.array_equal(x, x2))

# (c) 10 values without replacement
sample_10 = rng.choice(x, size=10, replace=False)
print("\n(c) 10 values without replacement:\n", sample_10)

# (d) 20 fair coin flips
flips = rng.integers(0, 2, size=20)   # high is exclusive → 0 or 1
print("\n(d) Flips:", flips)
print("    Number of heads (1s):", flips.sum())

(a) Sample mean: 50.2107
    Sample std:  10.3046

(b) Identical to first draw? True

(c) 10 values without replacement:
 [36.41907971 47.97301358 56.48898379 70.79242401 53.12478883 52.82649635
 43.49826055 51.74977623 35.88200693 65.49337529]

(d) Flips: [1 1 0 1 1 0 1 1 0 1 0 0 1 0 1 0 0 1 0 1]
    Number of heads (1s): 11


## Part 8: Reshaping, stacking, and transposing (10 points)

Let `m = np.arange(1, 13)`.

(a) Reshape `m` to 3×4 and then to 2×6 (use `-1` for one dimension in at least one).
(b) Transpose the 3×4 version.
(c) Stack the 3×4 matrix with itself vertically (`vstack`) and horizontally (`hstack`) and print the resulting shapes.
(d) Flatten the transposed matrix to 1-D.

In [33]:
m = np.arange(1, 13)

# (a) Reshape
m_3x4 = m.reshape(3, 4)
m_2x6 = m.reshape(2, -1)              # -1 lets NumPy infer the column count
print("(a) 3x4:\n", m_3x4)
print("\n    2x6 (using -1):\n", m_2x6)

# (b) Transpose 
m_T = m_3x4.T
print("\n(b) Transposed 3x4 → 4x3:\n", m_T)

# (c) Stacking
v = np.vstack([m_3x4, m_3x4])
h = np.hstack([m_3x4, m_3x4])
print("\n(c) vstack shape:", v.shape)   # (6, 4)
print("    hstack shape:", h.shape)     # (3, 8)

# (d) Flatten the transposed matrix
flat = m_T.flatten()
print("\n(d) Flattened transpose:\n", flat)

(a) 3x4:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

    2x6 (using -1):
 [[ 1  2  3  4  5  6]
 [ 7  8  9 10 11 12]]

(b) Transposed 3x4 → 4x3:
 [[ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]
 [ 4  8 12]]

(c) vstack shape: (6, 4)
    hstack shape: (3, 8)

(d) Flattened transpose:
 [ 1  5  9  2  6 10  3  7 11  4  8 12]


## Part 9: Sorting, uniqueness, and cumulative operations (10 points)

Let `v = np.array([4, 9, 4, 1, 7, 9, 3, 1, 4, 8])`.

(a) Print `v` sorted ascending and descending.
(b) Print the indices that would sort `v` (`argsort`).
(c) Print the unique values with their counts (`np.unique`).
(d) Print the cumulative sum of `v`, and the running maximum (`np.maximum.accumulate`).
(e) Print the 25th, 50th, and 75th percentiles of `v`.

In [34]:
v = np.array([4, 9, 4, 1, 7, 9, 3, 1, 4, 8])

# (a) Sorted ascending and descending
print("(a) Ascending: ", np.sort(v))
print("    Descending: ", np.sort(v)[::-1])

# (b) Indices that would sort v
idx = np.argsort(v)
print("\n(b) Argsort indices: ", idx) 
print("    Sorted via argsort:", v[idx]) 

# (c) Unique values with counts 
vals, counts = np.unique(v, return_counts=True) 
print("\n(c) Unique values:", vals)
print("    Counts:    ", counts)

# (d) Cumulative sum and running max
print("\n(d) Cumsum:            ", np.cumsum(v))
print("    Running maximum:   ", np.maximum.accumulate(v))

# (e) Percentiles 
print("\n(e) 25th / 50th / 75th percentiles:",
      np.percentile(v, [25, 50, 75]))

(a) Ascending:  [1 1 3 4 4 4 7 8 9 9]
    Descending:  [9 9 8 7 4 4 4 3 1 1]

(b) Argsort indices:  [3 7 6 0 2 8 4 9 1 5]
    Sorted via argsort: [1 1 3 4 4 4 7 8 9 9]

(c) Unique values: [1 3 4 7 8 9]
    Counts:     [2 1 3 1 1 2]

(d) Cumsum:             [ 4 13 17 18 25 34 37 38 42 50]
    Running maximum:    [4 9 9 9 9 9 9 9 9 9]

(e) 25th / 50th / 75th percentiles: [3.25 4.   7.75]


## Part 10: Linear algebra (10 points)

Let `A = np.array([[2., 1., -1.], [-3., -1., 2.], [-2., 1., 2.]])` and `b = np.array([8., -11., -3.])`.

(a) Compute the determinant of `A` and its inverse.
(b) Solve the system `A x = b` using `np.linalg.solve`.
(c) Verify the solution with `np.allclose(A @ x, b)`.
(d) Compute the element-wise product `A * A` and the matrix product `A @ A`; explain in one sentence the difference.

In [35]:
A = np.array([[2, 1, -1],
              [-3, -1, 2],
              [-2, 1, 2]])
b = np.array([8, -11, -3])

# (a) Determinant and inverse 
det = np.linalg.det(A)
A_inv = np.linalg.inv(A)
print("(a) Determinant:", round(det, 6))
print("    Inverse:\n", A_inv)

# (b) Solve A x = b
x = np.linalg.solve(A, b)
print("\n(b) Solution x =", x)

# (c) Verify
print("\n(c) A @ x matches b?", np.allclose(A @ x, b))
print("    A @ x =", A @ x)

# (d) Element-wise vs matrix product
elementwise = A * A
matmul     = A @ A
print("\n(d) Element-wise A * A:\n", elementwise)
print("\n    Matrix product A @ A:\n", matmul)

(a) Determinant: -1.0
    Inverse:
 [[ 4.  3. -1.]
 [-2. -2.  1.]
 [ 5.  4. -1.]]

(b) Solution x = [ 2.  3. -1.]

(c) A @ x matches b? True
    A @ x = [  8. -11.  -3.]

(d) Element-wise A * A:
 [[4 1 1]
 [9 1 4]
 [4 1 4]]

    Matrix product A @ A:
 [[  3   0  -2]
 [ -7   0   5]
 [-11  -1   8]]


## Part 11: Putting it together: a loop-free summary (10 points)

Using `spend_clean` and no loops, create a 2-D array `summary` of shape (5, 3) where row *j* contains, for month *j*, the **minimum, mean, and maximum** spending across customers. Then:

(a) Round `summary` to 1 decimal.
(b) Select only the months whose mean spending is above the overall mean of `spend_clean` (boolean row selection on `summary`) and print them.
(c) Compute the correlation matrix between months with `np.corrcoef` (use `rowvar=False`) and report the shape of the result.

In [36]:
# Build summary (5, 3): [min, mean, max] per month (column)
mins = spend_clean.min(axis=0)
means = spend_clean.mean(axis=0)
maxs = spend_clean.max(axis=0)

# Stack as columns to shape (5, 3)
summary = np.column_stack([mins, means, maxs])

# (a) Round to 1 decimal
summary = np.round(summary, 1)
print("(a) Summary (min, mean, max per month):\n", summary)

# (b) Months with mean above overall mean
overall_mean = spend_clean.mean()
mask = summary[:, 1] > overall_mean         # column 1 is the mean column
print("\n(b) Overall mean:", round(overall_mean, 2))
print("    Months with mean > overall mean:\n", summary[mask])

# (c) Correlation matrix
corr = np.corrcoef(spend_clean, rowvar=False)
print("\n(c) Correlation matrix shape:", corr.shape)
print("    (Expected: (5, 5) — one row/col per month)") 

(a) Summary (min, mean, max per month):
 [[ 70.  138.6 191. ]
 [ 37.  114.5 177. ]
 [ 31.   87.  149. ]
 [ 42.  126.6 189. ]
 [ 46.   88.  142. ]]

(b) Overall mean: 110.93
    Months with mean > overall mean:
 [[ 70.  138.6 191. ]
 [ 37.  114.5 177. ]
 [ 42.  126.6 189. ]]

(c) Correlation matrix shape: (5, 5)
    (Expected: (5, 5) — one row/col per month)
